# Context-Aware Music Evaluation and Recommendation Pipeline

Notebook tổng hợp cho báo cáo IT4142. Chạy lần lượt các cell để trình bày toàn bộ pipeline: data collection, cleaning, EDA, feature engineering, model evaluation, và recommendation demo.

## 1. Setup

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.load_data import load_default_tracks_dataset
from src.data.preprocess import preprocess_tracks
from src.features.audio_features import add_audio_helper_scores
from src.features.lyric_features import add_lyric_features
from src.features.pos_hmm_viterbi import add_pos_features
from src.features.emotion_features import create_weak_scenario_label
from src.visualization.eda_plots import pearson_correlation_matrix, chi_square_test
from src.models.scenario_ranker import ScenarioRanker
from src.models.hybrid_recommender import HybridRecommender
from src.models.scenario_classifier import ScenarioClassifier
from src.evaluation.metrics import recommendation_metrics
from src.app.streamlit_app import numeric_summary


## 2. Load Real or Sample Dataset

Ưu tiên `data/processed/tracks_app_ready.csv`, sau đó `data/raw/*.csv`/local CSV cũ, cuối cùng mới fallback `examples/sample_tracks.csv`.

In [ ]:
raw_df, source_path = load_default_tracks_dataset(max_rows=None)
print('Dataset source:', source_path)
print('Shape:', raw_df.shape)
raw_df.head()

## 3. Data Cleaning

In [ ]:
clean_df = preprocess_tracks(raw_df)
clean_df[['track_name', 'artist', 'genre', 'popularity']].head()

## 4. EDA and Relationship Analysis

In [ ]:
numeric_summary(clean_df).head(20)

In [ ]:
pearson_correlation_matrix(clean_df)

## 5. Feature Engineering: Audio, Lyrics, POS HMM/Viterbi, Scenario Labels

In [ ]:
features_df = add_audio_helper_scores(clean_df)
features_df = add_lyric_features(features_df)
features_df = add_pos_features(features_df)
features_df = create_weak_scenario_label(features_df)
features_df[['track_name', 'artist', 'scenario_label', 'energy', 'valence', 'tempo']].head()

In [ ]:
features_df['scenario_label'].value_counts()

In [ ]:
if {'genre', 'scenario_label'}.issubset(features_df.columns) and features_df['genre'].nunique() > 1:
    print(chi_square_test(features_df, 'genre', 'scenario_label'))
else:
    print('Not enough categorical variety for chi-square test.')

## 6. Machine Learning Model

In [ ]:
classifier = ScenarioClassifier(model_type='logistic_regression', tune=False)
if features_df['scenario_label'].nunique() > 1 and features_df['scenario_label'].value_counts().min() >= 2:
    metrics = classifier.evaluate_holdout(features_df)
else:
    classifier.fit(features_df)
    metrics = {'note': 'Class counts too small for holdout split; fitted for demo only.'}
metrics

## 7. Scenario-Based Recommendation Demo

In [ ]:
scenario = 'study'
ranker = ScenarioRanker().fit(features_df)
recommendations = ranker.recommend(scenario, top_k=10)
recommendations[['track_name', 'artist', 'scenario_score', 'explanations']].head(10)

In [ ]:
recommendation_metrics(recommendations, k=10)

## 8. Hybrid Recommendation Demo

In [ ]:
hybrid = HybridRecommender(classifier=None).fit(features_df)
hybrid_recs = hybrid.recommend('party', seed_song_names=[], top_k=10)
hybrid_recs[['track_name', 'artist', 'final_score', 'scenario_score', 'similarity_score', 'explanations']].head(10)

## 9. Notes for Report

- Nếu đang dùng weak labels, ghi rõ `scenario_label` được sinh bằng heuristic audio/lyrics.
- Không commit dataset lớn; chỉ commit sample data.
- Với dataset Kaggle thật, chạy `python scripts/prepare_real_data.py --input data/raw/<file>.csv` trước khi mở app/notebook.